# Counterfactual Generation with TabPFN

This notebook demonstrates the full pipeline for training a TabPFN transformer
to generate counterfactual explanations:

1. **Data generation** — SCM-based factual/counterfactual pairs
2. **Training** — adapt TabPFN to predict counterfactual deltas
3. **Evaluation** — measure validity, proximity, and sparsity
4. **Analysis** — what works, what doesn't, next steps

**Core idea**: Instead of `f(dataset, query_x) → class`, we train
`f(dataset, query_x, target_label) → delta_x` where `query_x + delta_x`
is the counterfactual point.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cpu"
print(f"Using device: {device}")

## 1. Data Generation

We use `CounterfactualSCMGenerator` to create factual-counterfactual pairs.
Each pair comes from a randomly sampled Structural Causal Model (MLP with
random weights). The generator perturbs input features and propagates
changes causally through the SCM.

In [ ]:
from tabpfn.priors.counterfactual import (
    CounterfactualSCMGenerator,
    get_default_counterfactual_config,
)

config = get_default_counterfactual_config()
gen = CounterfactualSCMGenerator(config, device=device)

num_features = 5
batch = gen.generate_batch(batch_size=1, seq_len=128, num_features=num_features)

print(f"x_factual shape:        {batch.x_factual.shape}")
print(f"x_counterfactual shape: {batch.x_counterfactual.shape}")
print(f"y_factual_class shape:  {batch.y_factual_class.shape}")
print(f"label_flipped shape:    {batch.label_flipped.shape}")
print(f"Label flip rate:        {batch.label_flipped[:, 0].float().mean():.2f}")

In [ ]:
# Visualize factual vs counterfactual for a single SCM dataset
b = 0  # batch element
x_f = batch.x_factual[:, b].numpy()
x_cf = batch.x_counterfactual[:, b].numpy()
flipped = batch.label_flipped[:, b].numpy().astype(bool)
deltas = x_cf - x_f

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: First two features, factual vs counterfactual
ax = axes[0]
ax.scatter(x_f[:, 0], x_f[:, 1], c="blue", alpha=0.4, label="Factual", s=20)
ax.scatter(x_cf[flipped, 0], x_cf[flipped, 1], c="red", alpha=0.4, label="CF (flipped)", s=20)
for i in np.where(flipped)[0][:20]:  # arrows for first 20 flipped
    ax.annotate("", xy=(x_cf[i, 0], x_cf[i, 1]), xytext=(x_f[i, 0], x_f[i, 1]),
                arrowprops=dict(arrowstyle="->", color="gray", alpha=0.3))
ax.set_xlabel("Feature 0")
ax.set_ylabel("Feature 1")
ax.set_title("Factual vs Counterfactual (features 0-1)")
ax.legend()

# Plot 2: Delta distribution per feature
ax = axes[1]
for feat in range(num_features):
    ax.hist(deltas[flipped, feat], bins=30, alpha=0.5, label=f"feat {feat}")
ax.set_xlabel("Delta value")
ax.set_ylabel("Count")
ax.set_title("Delta distribution (label-flipped samples)")
ax.legend(fontsize=8)

# Plot 3: Intervention sparsity
ax = axes[2]
intervention_mask = batch.intervention_mask[:, b].numpy()
sparsity_per_sample = 1.0 - intervention_mask.mean(axis=1)
ax.hist(sparsity_per_sample, bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Fraction of unchanged features")
ax.set_ylabel("Count")
ax.set_title(f"Sparsity (mean={sparsity_per_sample.mean():.2f})")

plt.tight_layout()
plt.show()

## 2. Training

We train a small TabPFN transformer with MSE loss to predict counterfactual
deltas. The model receives context (factual x, factual y) and queries
(factual x, target y), then predicts `delta = x_cf - x_factual`.

In [ ]:
from tabpfn.train_counterfactual import train_counterfactual, TOY_CONFIG
import inspect

# Use toy config with reduced epochs for notebook demo
# Filter to only keys accepted by train_counterfactual
valid_params = set(inspect.signature(train_counterfactual).parameters.keys())
train_config = {k: v for k, v in TOY_CONFIG.items() if k in valid_params}
train_config["epochs"] = 20
train_config["steps_per_epoch"] = 50
train_config["verbose"] = True

print("Training config:")
for k, v in train_config.items():
    print(f"  {k}: {v}")

In [ ]:
loss_history = []

def epoch_callback(model, progress):
    """Callback to capture training progress."""
    pass

total_loss, model, dl, loss_history = train_counterfactual(
    **train_config, epoch_callback=epoch_callback
)
print(f"\nFinal MSE loss: {total_loss:.4f}")

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(loss_history) + 1), loss_history, "b-o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training Loss Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss reduction: {loss_history[0]:.4f} -> {loss_history[-1]:.4f} "
      f"({(1 - loss_history[-1]/loss_history[0])*100:.1f}% decrease)")

## 3. Evaluation

We evaluate the trained model on fresh test SCM datasets using four metrics:
- **Delta MSE**: How close are predicted deltas to ground-truth deltas?
- **Validity**: Does the predicted counterfactual actually change the label?
- **Proximity**: How far does the counterfactual move from the original point?
- **Sparsity**: What fraction of features remain unchanged?

In [ ]:
from tabpfn.eval_counterfactual import evaluate

num_test_datasets = 50
metrics = evaluate(
    model,
    num_test_datasets=num_test_datasets,
    num_features=num_features,
    seq_len=128,
    device=device,
)

## 4. Example Predictions

Let's look at individual predictions to understand model behavior.

In [ ]:
from tabpfn.eval_counterfactual import generate_test_data, run_inference

# Generate a small test set for visualization
test_batches, single_eval_pos = generate_test_data(
    num_datasets=10, seq_len=128, num_features=num_features, device=device
)
pred_deltas, true_deltas, query_x, target_labels = run_inference(
    model, test_batches, single_eval_pos, device=device
)

# Show a few examples
n_examples = 5
print("Example predictions (first 5 query points):\n")
for i in range(n_examples):
    print(f"--- Example {i+1} ---")
    print(f"  Query x:       {query_x[i].numpy().round(3)}")
    print(f"  Target label:  {target_labels[i].item():.0f}")
    print(f"  True delta:    {true_deltas[i].numpy().round(3)}")
    print(f"  Pred delta:    {pred_deltas[i].numpy().round(3)}")
    print(f"  True CF:       {(query_x[i] + true_deltas[i]).numpy().round(3)}")
    print(f"  Pred CF:       {(query_x[i] + pred_deltas[i]).numpy().round(3)}")
    print()

In [ ]:
# Visualize predicted vs true deltas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Predicted vs true delta (scatter per feature)
ax = axes[0]
for feat in range(num_features):
    ax.scatter(
        true_deltas[:, feat].numpy(), pred_deltas[:, feat].numpy(),
        alpha=0.2, s=10, label=f"feat {feat}"
    )
lim = max(abs(true_deltas.min().item()), abs(true_deltas.max().item()),
          abs(pred_deltas.min().item()), abs(pred_deltas.max().item()))
ax.plot([-lim, lim], [-lim, lim], "k--", alpha=0.3, label="y=x")
ax.set_xlabel("True delta")
ax.set_ylabel("Predicted delta")
ax.set_title("Predicted vs True Delta")
ax.legend(fontsize=7)

# Plot 2: Per-feature MSE
ax = axes[1]
per_feature_mse = ((pred_deltas - true_deltas) ** 2).mean(dim=0).numpy()
ax.bar(range(num_features), per_feature_mse, color="steelblue", edgecolor="white")
ax.set_xlabel("Feature")
ax.set_ylabel("MSE")
ax.set_title("Per-Feature MSE")
ax.set_xticks(range(num_features))

# Plot 3: Delta magnitude distribution (predicted vs true)
ax = axes[2]
true_norms = true_deltas.norm(dim=-1).numpy()
pred_norms = pred_deltas.norm(dim=-1).numpy()
ax.hist(true_norms, bins=30, alpha=0.5, label="True", color="blue")
ax.hist(pred_norms, bins=30, alpha=0.5, label="Predicted", color="red")
ax.set_xlabel("L2 norm of delta")
ax.set_ylabel("Count")
ax.set_title("Delta Magnitude Distribution")
ax.legend()

plt.tight_layout()
plt.show()

## 5. Analysis

### What works
- The TabPFN architecture adapts naturally to counterfactual generation by
  changing the output head from class logits to continuous deltas.
- The in-context learning paradigm lets the model see examples from a specific
  SCM and generate counterfactuals for that same SCM without retraining.
- The data loader correctly prioritizes label-flipped samples for query positions.

### What doesn't (limitations of this toy experiment)
- **Small scale**: 5 features, 128 samples, 4-layer transformer — real datasets
  will need larger models and longer training.
- **Approximate validity**: Without retaining the original SCM classifier, we
  use a heuristic validity check (non-trivial delta norm).
- **Single perturbation strategy**: Only additive noise is used; gradient-guided
  or marginal replacement might produce more realistic counterfactuals.

### Next steps
1. Scale to more features (10-50) and larger model architectures
2. Implement true validity checking by retaining SCM classifiers during evaluation
3. Compare with baseline counterfactual methods (DiCE, Wachter, etc.)
4. Add actionability constraints (some features may not be changeable)
5. Test on real tabular datasets with known causal structure